In [6]:
import pandas as pd


df = pd.read_csv("magaza_yorumlari_duygu_analizi.csv",encoding="utf-16")
df.head()

,Görüş,Durum
0,"ses kalitesi ve ergonomisi rezalet, sony olduğ...",Olumsuz
1,hizli teslimat tesekkürler,Tarafsız
2,ses olayı süper....gece çalıştır sıkıntı yok.....,Olumlu
3,geldi bigün kullandık hemen bozoldu hiçtavsiye...,Olumsuz
4,Kulaklığın sesi kaliteli falan değil. Aleti öv...,Olumsuz


In [7]:
df.columns = ["text", "label"]

df.dropna(inplace=True)

# Etiketleri küçük harfe çevir
df["label"] = df["label"].str.lower()

# Mevcut sınıflar
print(df["label"].value_counts())


print(df.head())

olumlu      4252
olumsuz     4237
tarafsız    2937
Name: label, dtype: int64
                                                text     label
0  ses kalitesi ve ergonomisi rezalet, sony olduğ...   olumsuz
1                         hizli teslimat tesekkürler  tarafsız
2  ses olayı süper....gece çalıştır sıkıntı yok.....    olumlu
3  geldi bigün kullandık hemen bozoldu hiçtavsiye...   olumsuz
4  Kulaklığın sesi kaliteli falan değil. Aleti öv...   olumsuz


In [8]:
df = df[df["label"].isin(["olumlu", "olumsuz"])]

# Kalan sınıfları tekrar kontrol et
print(df["label"].value_counts())

# Örnek veri
df.sample(5)

olumlu     4252
olumsuz    4237
Name: label, dtype: int64


,text,label
3398,Bu ürünle ilgili olarak az önce ikinci sipariş...,olumlu
1248,Olmamış,olumsuz
7543,Ps4 pro kapasitesini yükseltmek için aldım kar...,olumsuz
7482,Hızlı ve çok özenli kargo çok teşekkür ederim....,olumlu
8348,Ürüne yorum yapamıyorum ama montaj hizmeti sıf...,olumsuz


In [9]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

def clean_text(text):
    text = text.lower()  # küçük harfe çevir
    text = re.sub(r"[^\w\s]", "", text)  # noktalama kaldır
    return text

# Temizlenmiş metinler
df["clean_text"] = df["text"].apply(clean_text)


custom_stop_words = [
    "ve", "ile", "de", "da", "bu", "bir", "şu", "o", "mı", "mi",
    "ki", "ne", "veya", "ya", "ya da", "gibi", "ise", "çünkü", "ama", "ancak"
]


vectorizer = TfidfVectorizer(stop_words=custom_stop_words, max_features=3000)
X = vectorizer.fit_transform(df["clean_text"])

In [11]:
y = df["label"]

print("Vektör boyutu:", X.shape)
print("Örnek etiketler:", y.unique())

Vektör boyutu: (8489, 3000)
Örnek etiketler: ['olumsuz' 'olumlu']


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# Eğitim/Test bölme
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Naive Bayes modeli
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_preds = nb_model.predict(X_test)

# SVM modeli
svm_model = LinearSVC()
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)

# Raporlama
print(" Naive Bayes Raporu")
print(classification_report(y_test, nb_preds))

print("\n SVM Raporu")
print(classification_report(y_test, svm_preds))

 Naive Bayes Raporu
              precision    recall  f1-score   support

      olumlu       0.87      0.92      0.89       859
     olumsuz       0.91      0.87      0.89       839

    accuracy                           0.89      1698
   macro avg       0.89      0.89      0.89      1698
weighted avg       0.89      0.89      0.89      1698


 SVM Raporu
              precision    recall  f1-score   support

      olumlu       0.88      0.89      0.89       859
     olumsuz       0.89      0.88      0.88       839

    accuracy                           0.89      1698
   macro avg       0.89      0.89      0.89      1698
weighted avg       0.89      0.89      0.89      1698



In [19]:
yorumlar = [
    "Ürün harika, kargom çok hızlı geldi ve paketleme mükemmeldi!",  
    "Ürün çok kötü, paramı resmen çöpe attım, berbat bir deneyimdi." 
]

In [20]:
# Yorumları temizle
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text

yorumlar_clean = [preprocess_text(yorum) for yorum in yorumlar]

# TF-IDF vektörleştirme (önceden eğittiğimiz vectorizer kullanılıyor)
yorumlar_vectorized = vectorizer.transform(yorumlar_clean)

# Naive Bayes tahmini
nb_tahminler = nb_model.predict(yorumlar_vectorized)

# SVM tahmini
svm_tahminler = svm_model.predict(yorumlar_vectorized)

# Sonuçları yazdıralım
for i, yorum in enumerate(yorumlar):
    print(f"Yorum: {yorum}")
    print(f"Naive Bayes Tahmini: {nb_tahminler[i]}")
    print(f"SVM Tahmini: {svm_tahminler[i]}")
    print("-" * 60)

Yorum: Ürün harika, kargom çok hızlı geldi ve paketleme mükemmeldi!
Naive Bayes Tahmini: olumlu
SVM Tahmini: olumlu
------------------------------------------------------------
Yorum: Ürün çok kötü, paramı resmen çöpe attım, berbat bir deneyimdi.
Naive Bayes Tahmini: olumsuz
SVM Tahmini: olumsuz
------------------------------------------------------------
